# (1) Install dependencies

In [1]:
!pip install fastapi uvicorn langgraph sentence-transformers chromadb pydantic


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.6 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found 

# (2) Prepare the corpus (8 docs)

In [2]:
import os

os.makedirs("docs", exist_ok=True)

docs = {
    "doc_01.txt": "Zepto delivers grocery and household essentials ...",
    "doc_02.txt": "Grocery and perishable items may be reported ...",
    "doc_03.txt": "Zepto offers three account tiers ...",
    "doc_04.txt": "Every Zepto order shows a live rider-tracking map ...",
    "doc_05.txt": "Orders can be cancelled free of cost ...",
    "doc_06.txt": "If an order arrives with damaged, spoiled, or missing items ...",
    "doc_07.txt": "Zepto gift cards are available in fixed denominations ...",
    "doc_08.txt": "Zepto customer support is available via in-app chat ..."
}

for fname, content in docs.items():
    with open(f"docs/{fname}", "w") as f:
        f.write(content)


# (3) Chunk + Embed + Store in ChromaDB

In [3]:
import os
from sentence_transformers import SentenceTransformer
import chromadb

# Load model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Initialize ChromaDB (persistent so it survives runtime restarts)
client = chromadb.PersistentClient(path="embeddings/chromadb_store")

# Create or get collection
collection = client.get_or_create_collection("zepto_policies")

# Clear collection if re-running
existing = collection.get()
if existing["ids"]:
    collection.delete(ids=existing["ids"])

# Chunk and embed
for fname in os.listdir("docs"):
    with open(f"docs/{fname}") as f:
        text = f.read()
    chunks = [text[i:i+300] for i in range(0, len(text), 300)]
    for idx, chunk in enumerate(chunks):
        emb = model.encode(chunk).tolist()
        collection.add(
            ids=[f"{fname}_{idx}"],
            embeddings=[emb],
            documents=[chunk]
        )

results = collection.query(query_texts=["What is Zepto’s delivery fee?"], n_results=3)
print("Retrieved chunks:", results["documents"])
print("Source IDs:", results["ids"])


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 104MiB/s]


Retrieved chunks: [['Zepto delivers grocery and household essentials ...', 'Zepto customer support is available via in-app chat ...', 'Zepto gift cards are available in fixed denominations ...']]
Source IDs: [['doc_01.txt_0', 'doc_08.txt_0', 'doc_07.txt_0']]


# (4) Structured prompt template

In [4]:
prompt_template = """
Role: You are Zepto’s support assistant.
Context: Use only the retrieved policy text.
Task: Answer clearly and concisely.
Format: JSON with fields answer, sources, confidence.
Length: Keep under 150 words.
Constraint: Do not answer using information not present in the provided context.

Example:
Q: What is Zepto’s delivery fee?
A: {"answer":"Orders below INR 149 incur INR 25 fee","sources":["doc_01_0"],"confidence":1.0}
"""


# (5) LangGraph StateGraph (mock + real toggle)

In [5]:
import os
MOCK_LLM = os.getenv("MOCK_LLM", "1")  # default mock

def classify_intent(query: str):
    keywords = ["delivery","return","refund","membership","tracking","cancel","gift card","support hours"]
    if any(k in query.lower() for k in keywords):
        return "policy_question"
    return "general_question"

def retrieve_and_answer(query: str):
    results = collection.query(query_texts=[query], n_results=3)
    top_chunk = results["documents"][0][0]
    if MOCK_LLM == "1":
        return {"answer": f"Based on the retrieved context: {top_chunk[:200]}",
                "sources": results["ids"][0],
                "confidence": 1.0}
    else:
        # Real LLM call placeholder
        return {"answer": "Real LLM answer here", "sources": results["ids"][0], "confidence": 0.9}

def direct_answer(query: str):
    return {"answer": "This is a general answer (mock).",
            "sources": [],
            "confidence": 1.0}


# (6) FastAPI wrapper

In [6]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class AskRequest(BaseModel):
    query: str

class AskResponse(BaseModel):
    answer: str
    sources: list
    confidence: float

@app.post("/ask", response_model=AskResponse)
def ask(req: AskRequest):
    intent = classify_intent(req.query)
    if intent == "policy_question":
        return retrieve_and_answer(req.query)
    else:
        return direct_answer(req.query)


#(7) Run locally in Colab

In [7]:
%%writefile main.py
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class AskRequest(BaseModel):
    query: str

class AskResponse(BaseModel):
    answer: str
    sources: list
    confidence: float

@app.post("/ask", response_model=AskResponse)
def ask(req: AskRequest):
    # Simple mock response for testing
    return {"answer": "Hello from FastAPI!", "sources": [], "confidence": 1.0}

!uvicorn main:app --host 0.0.0.0 --port 7860 --reload


Writing main.py


# (8)  Example calls

In [8]:
print(ask(AskRequest(query="What is Zepto’s delivery fee?")))
print(ask(AskRequest(query="Who is the CEO of Zepto?")))


{'answer': 'Based on the retrieved context: Zepto delivers grocery and household essentials ...', 'sources': ['doc_01.txt_0', 'doc_08.txt_0', 'doc_07.txt_0'], 'confidence': 1.0}
{'answer': 'This is a general answer (mock).', 'sources': [], 'confidence': 1.0}
